# STAT 301 Case Study Project - Wildfire

**Name:** Arnav Dhablania

**Group Number & Name:** 33 - 301byte

**Group Members:** Antarip Kashyap, Arnav Dhablania, Eleanor Lam 

## Planning Stage: Data Description & Exploratory Data Analysis (Individual Assignment 1)

In [ ]:
library(tidyverse)
library(diversedata)
library(broom)
library(dplyr)
library(car)

### 1. Data Description

In [ ]:
# hi
data("wildfire")

**a) Number of observations and variables**

In [ ]:
dim(wildfire)

There are 26,551 observations and 35 variables in the dataset.

**b) Table of variable names and type**

In [ ]:
vars_table <- tibble(Variable_Name = names(wildfire), Data_Type = sapply(wildfire, class))
print(vars_table, n = 35)

**c) Data collection process**

The dataset contains historical wildfire data from Alberta from 2006 to 2024 to monitor and assess wildfire risks. The collection process for the data probably included on-ground assessments by fire crews, meteorological sensors in the forests or maybe even spatial tracking.

**d) Data Source and Citation**

<u>Source:</u> The data is sourced from the Government of Alberta and is accessed through the Government of Canada’s Open Government Portal.

<u>Citation:</u> Government of Alberta. Historical wildfire data: 2006-2024. Sourced via the Government of Canada’s Open Government Portal, available under the Open Government Licence - Alberta.

### 2. Questions

**a) Question of Interest**

How well can a multiple linear regression model predict the final burn area of a wildfire? Variables that can be used to predict this could be initial temperature, wind speed, relative humidity, and fire spread rate.

**b) Prediction vs Inference**

This question focuses on both prediction and inference. We could be predicting the size of a wildfire to help with future wildfires, whereas we could use inference to find which environmental factor has the most significant impact on the scale of a wildfire.

**c) Response Variable**

The response variable for this model is current_size which is a continuous numeric variable and holds the estimated area burned by the wildfire.

**d) Covariates**

I think that there are four main covariates that can be used for this model. "wind_speed" and "fire_spread_rate" will likely be the main variables used to predict the ability of the fire to spread. "temperature" and "relative_humidity" are good variables that could be used to predict the scale of a wildfire using the conditions of the environment before the fire began.

**e) Other Variables**

Other variables in the dataset also provide a lot of insight into the type of wildfire and could be used to classify the rate of spread depending on the forest type. Example, "fuel_type" could help us classify the rate of spread depending on the vegetation.

### 3. Exploratory Data Analysis and Visualization

**a) Load Dataset**

I have already loaded the dataset in the setup phase, but I will load the dataset again for a consistent flow.

In [ ]:
data("wildfire", package = "diversedata")
head(wildfire, 3)

**b) Data Cleaning and Wrangling**

For this process, I will only select variables relevant to our question and I will convert the continuous weather variables to numeric and remove any incomplete observations that would break the regression model. 

In [ ]:
wildfire_clean <- wildfire %>%
    select(current_size, temperature, wind_speed, relative_humidity, fire_spread_rate, fuel_type, ia_access) %>%
    mutate(temperature = as.numeric(temperature), wind_speed = as.numeric(wind_speed), relative_humidity = as.numeric(relative_humidity))
dim(wildfire_clean)
head(wildfire_clean)

**c) NA Values**

In [ ]:
missing_summary <- tibble(Variable = names(wildfire), Missing_num = colSums(is.na(wildfire)), Percent_Missing = (Missing_num / nrow(wildfire)) * 100) %>%
    arrange(desc(Percent_Missing))
head(missing_summary, 10)

**Interpretation:** ia_arrival_at_fire_date and fire_fighting_start_date have large proportions of data missing at 29% and 28.5% respectively.

**d) Class Imbalance**

I am using the "general_cause" variable to look for class imbalances

In [ ]:
imbalance_check <- wildfire %>%
    count(general_cause) %>%
    mutate(Percentage = n/sum(n) * 100) %>%
    arrange(desc(Percentage))
imbalance_check

**Interpretation:** Yes there is a clear class imbalance, the top two causes (Lightning and Recreation) account for ~55% of the data. While the bottom 4 causes barely make up 1% of the data.

**e) Exploratory Visualization**

The facet scatterplot below shows the relationship between our response variable and two of our main meteorological covariates, while observing how the initial attack access method might cause differences. 

In [ ]:
plot <- ggplot(wildfire_clean, aes(x = temperature, y= current_size, color = wind_speed)) + 
    geom_point(alpha = 0.5) + 
    facet_wrap(~ ia_access) +
    theme_minimal() + 
    labs(title = "Temperature and wind speed impact on wildfire size", x = "Temperature", y = "Final burned area", color = "Wind speed")
options(repr.plot.width = 15, repr.plot.height = 12)
plot

**Interpretation:** The faceted scatterplot shows the final burned area against the temperature and wind speed, split by the initial attack access method. The plot shows that majority of the wildfires remain small, rare massive outlier fires happen mostly at higher temperatures. Access Method here shows us that most extreme fires occur when access is "Conventional R/W" and "Unknown". This plot directly addresses our research question by exploring how specific weather conditions and access methods influence the fire size.

**f) AI Tool Disclosure**

I used an AI tool (gemini) just to get some inspiration for the wording of my written interpretations, to double-check the ggplot functions, and help with the citation.

## Methods and Plan & Computational Code and Output (Individual Assignment 2)

Note: Feedback for assignment 1 included not needing to drop NA values in assignment 1 and increasing the size of the plot. I have done both at their respective questions.

### 1) Methods and Plan

**a) Review Question**

In assignment part 1, my question aimed to answer how well a multiple linear regression model can predict the burn area of a fire. For assignment 2, I am refining this question to better align with the observational nature of the data. The new question is "Which combination of variables from the dataset are the most useful predictors of a wildfire's final burned area?" This question allows us to keep focus on predicting the fire size, but now accounting for all variables in the dataset instead of handpicking only weather variables. With the help of variable selection techniques(LASSO or stepwise), we can identify the most relevant variables that actually affect the burned area size, instead of restricting the model.

**b) Method**

To address the question of interest, I plan on using Multiple Linear Regression combined with Backward Stepwise Selection. This is appropriate since our response variable is continuous (current_size) and we want to evaluate how multiple explanatory variables together affect the response. Since the goal is to now identify the most useful predictors without handpicking them, backward selection is the perfect choice as it starts with a full model with all variables and systematically drops the least useful ones based on AIC to find the most optimal model. Also, since the data is right-skewed, I will be applying a log transformation to the response variable to ensure that the model satisfies all assumptions for regression. 

**c) Assumptions and Limitations**

**Assumptions:** 

Applying a multiple linear regression with backward stepwise selection requires several key assumptions. First, the relationship between the log-transformed response (wildfire size) and selected predictors must be linear. Second, the observations must be independent, meaning the size of one recorded wildfire should not influence another. Third, the variance of the residuals should be approximately constant across all fitted values. Fourth, the residuals should also approximately be normally distributed. Finally, there should be low multicollinearity among predictors.

**Limitations:** 

A limitation of this analysis is that it relies on observational data, meaning our final model can be used to find associations only and not for any causal effects. Additionally, backward stepwise selection relies purely on statistical metrics like AIC to drop variables; it may disregard a predictor that has a strong influence on the fire size in the real world, but it doesn't contribute strongly enough mathematically in this specific sample. A problem we might have is that weather metrics such as temperature and windspeed are measured at a specific time and don't account for weather pattern changes during the fire, which could cause extra variability. Wildfires often occur in clusters, which violates the independence assumption. There is also some correlation between weather variables like high heat and low humidity, which adds some multicollinearity.  Missing values may also reduce the sample size and potentially bias the analysis if the missingness is not random.

**Verifying Assumptions:**

- Linearity: Examined with a residuals vs fitted values plot. If the linear assumption holds, the residuals would be randomly scattered around the line without any specific patterns.

- Independence: We can partially verify this by checking the fire_number to check for duplicate records, but to assess the clustering problem, we must remain mindful of spatial groupings of fires using variables like longitudes, latitudes, and their start dates.

-  Homoscedasticity: This is also checked using the residuals vs fitted plot. The vertical spread of residuals should be constant throughout the fitted values. If it creates a funnel-like shape, it shows Heteroscedasticity, meaning our variance is not constant.

-  Multicollinearity: This can be solved using VIF. If the VIF value is low, the predictors are not too collinear, but if the VIF value is large, we might have to drop any overlapping variables.

-  Normality: Using a Q-Q plot and residual histogram, if the points are along the diagonal reference line in the Q-Q plot and have a bell-shaped histogram, the errors are normally distributed. Using a log transformation on our response variable should help with the alignment. 

### 2) Computational Code and Output

**a) Model Implementation**

In this section, I prepare the dataset by filtering out any missing values for our response and predictor variables to make sure that the stepwise model fits our data well. I then applied a log transformation to the response variable and added 1 to address the extreme right-skewness of the fire sizes. I then fit a full multiple linear regression model that contains all potential geographic, environmental, and categorical predictors and predicts the transformed wildfire size accordingly. I applied backward stepwise selection using AIC to drop the least useful variables. In the end, I calculate the VIF on this chosen model to verify that the remaining predictors do not violate multicollinearity. 

In [ ]:
model_data <- wildfire %>%
    filter(!is.na(current_size), !is.na(temperature), !is.na(relative_humidity), !is.na(wind_speed), !is.na(latitude), !is.na(longitude), !is.na(year), !is.na(general_cause)) %>%
    mutate(log_current_size = log(current_size + 1), general_cause = factor(general_cause))

full_model <- lm(log(current_size + 1) ~ temperature + wind_speed + relative_humidity + fire_spread_rate + latitude + longitude + year + general_cause, data = model_data)
stepwise_model <- step(full_model, direction = "backward", trace = 0)
model_vif <- vif(stepwise_model)

summary(stepwise_model)

**b) Table of Results**

To report the results of the multiple linear regression, I have generated a summary table containing the estimated coefficients, 95% confidence intervals, and p-values for each predictor that remained after the backward stepwise selection. This allows us to easily identify the direction, magnitude, and statistical significance of each weather variable's relationship with the log-transformed wildfire size.

In [ ]:
table <- tidy(stepwise_model, conf.int = TRUE) %>%
    mutate(estimate = round(estimate, 4), conf.low = round(conf.low, 4), conf.high = round(conf.high, 4), p.value = round(p.value, 4)) %>%
    select(term, estimate, conf.low, conf.high, p.value) %>%
    arrange(desc(estimate))
table

**c) Interpretation of Results**

The backward stepwise selection identified fire spread rate, wind speed, relative humidity, longitude, latitude, year, and general cause as the most useful predictors of log-transformed wildfire size. Wind speed and fire spread rate show a positive association with fire size, and temperature was completely dropped from the stepwise model, maybe because its effect was already accounted for by geographical variables. These questions directly address our research question by showing us that the fire's final area is best predicted by a combination of weather, spatial, and fire source data rather than handpicking a few variables. A potential problem is that the general cause adds many statistically insignificant levels like "Government" or "Forest Industry", which could be fixed by grouping such minor categories into a single group, "Other", to simplify the model.

**AI disclosure:** AI was only used to fix my grammar in the written portions of assignment 2. All other work including the idea and concept is my own.